    # Notebook 4 — Le CAPM à l'épreuve des données

    **Le Quotient Financier · Travaux pratiques d'histoire de la gestion de portefeuille**

    En 1964, William Sharpe pousse son modèle de 1963 jusqu'à l'équilibre : si tous les investisseurs
    optimisent comme Markowitz, quels prix en résultent ? La réponse, le **CAPM** (*Capital Asset Pricing Model*,
    ou Medaf en français), lui vaudra le prix Nobel en 1990. Ce notebook expose la théorie, puis la confronte
    aux données, hypothèse par hypothèse.

    **Au programme**

    1. La théorie, et une mise au point de vocabulaire.
    2. Alpha et bêta de 30 titres.
    3. Tester la droite de marché des titres.
    4. Les hypothèses à l'épreuve : normalité, stabilité des bêtas, portefeuille de marché, emprunt.
    5. L'arithmétique de la gestion active (Sharpe, 1991).
    6. Ce qu'il en reste.

> *Ce notebook est un support pédagogique du Quotient Financier. Il illustre des méthodes historiques
> et académiques ; il ne constitue en aucun cas un conseil en investissement ni une recommandation
> d'achat ou de vente de titres. Les titres utilisés servent uniquement d'exemples.*

## 1. La théorie en une page

**Le théorème de séparation (Tobin, 1958).** Quand on peut prêter et emprunter au taux sans risque $r_f$,
tout investisseur détient la même combinaison d'actifs risqués, le portefeuille tangent du notebook 1.
Seul le dosage entre ce portefeuille et l'actif sans risque dépend de son aversion au risque.

**L'équilibre (Sharpe, 1964 ; Lintner, 1965 ; Mossin, 1966).** Si tous les investisseurs ont les mêmes anticipations,
ils détiennent tous le même portefeuille tangent. Or, à l'équilibre, l'ensemble des investisseurs détient
forcément tous les actifs existants. Le portefeuille tangent est donc le **portefeuille de marché**.

**La droite de marché des titres.** Le rendement espéré de chaque titre ne dépend alors que de son bêta :

$$\mathbb{E}[R_i] - r_f = \beta_i \left(\mathbb{E}[R_M] - r_f\right), \qquad \beta_i = \frac{\operatorname{Cov}(R_i, R_M)}{\operatorname{Var}(R_M)}$$

Le risque propre, diversifiable, n'est pas rémunéré. Dans le modèle de Sharpe de 1963, $R_i = a_i + b_i I + e_i$ :
le CAPM prédit que la constante, une fois les rendements exprimés au-delà du taux sans risque, est nulle.
C'est elle qu'on appelle **l'alpha de Jensen**.

**Les hypothèses.** Investisseurs qui ne regardent que moyenne et variance, sur un horizon unique, avec les mêmes
anticipations ; prêt et emprunt illimités au même taux sans risque ; pas de frais ni d'impôts ;
tous les actifs négociables et divisibles.

> **Mise au point de vocabulaire.** Trois notions portent des noms voisins et sont souvent confondues.
> - Un portefeuille **efficient**, au sens de Markowitz, est un portefeuille de la frontière : aucun autre ne rapporte
>   plus pour le même risque. Il y en a une infinité.
> - Le portefeuille **optimal** est celui que choisit un investisseur donné, selon son aversion au risque,
>   ou le portefeuille tangent quand on raisonne avec un actif sans risque.
> - **L'efficience des marchés**, au sens d'Eugene Fama, est une tout autre idée : les prix reflètent
>   l'information disponible.
>
> Dire que « le portefeuille de marché est efficient » est une affirmation au sens de Markowitz, pas de Fama.
> Richard Roll (1977) a montré que tester le CAPM revient exactement à tester cette efficience-là.

## 0. Mise en place

In [ ]:
# Sur Google Colab ou une installation neuve, décommentez la ligne suivante :
# !pip install yfinance numpy pandas scipy matplotlib

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # masque les avertissements de polices
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "mathtext.fontset": "dejavusans"})   # formules des axes (ex. 10^1) avec la police par défaut
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.set_printoptions(precision=4, suppress=True)

SEMAINES = 52              # nombre de semaines par an, pour annualiser
TAUX_SANS_RISQUE = 0.02    # taux sans risque annuel retenu (hypothèse simplificatrice)

In [ ]:
from scipy import stats

# 30 titres américains de 9 secteurs, l'ETF S&P 500 et un ETF obligataire américain agrégé
TITRES = {
    "AAPL": "Technologie", "MSFT": "Technologie", "ORCL": "Technologie", "CSCO": "Technologie", "INTC": "Technologie",
    "JPM": "Finance", "BAC": "Finance", "WFC": "Finance", "GS": "Finance",
    "XOM": "Énergie", "CVX": "Énergie", "COP": "Énergie",
    "JNJ": "Santé", "PFE": "Santé", "MRK": "Santé", "UNH": "Santé",
    "KO": "Consommation de base", "PG": "Consommation de base", "PEP": "Consommation de base", "WMT": "Consommation de base",
    "CAT": "Industrie", "HON": "Industrie", "MMM": "Industrie",
    "NEE": "Services publics", "DUK": "Services publics", "SO": "Services publics",
    "HD": "Consommation cyclique", "MCD": "Consommation cyclique", "NKE": "Consommation cyclique",
    "VZ": "Télécoms",
}
MARCHE, OBLIGATIONS = "SPY", "AGG"
SECTEURS = {**TITRES, MARCHE: "Marché", OBLIGATIONS: "Obligations"}
TOUS = list(TITRES) + [MARCHE, OBLIGATIONS]
DEBUT, FIN = "2015-01-01", "2024-12-31"
FICHIER = "donnees_prix_capm.csv"

In [ ]:
def donnees_simulees(tickers, secteurs, debut, fin, graine=42):
    """Prix hebdomadaires simulés par un modèle à facteurs (marché + secteurs + bruit à queues épaisses).
    Utilisés uniquement si le téléchargement échoue, pour que le notebook reste exécutable."""
    rng = np.random.default_rng(graine)
    dates = pd.date_range(debut, fin, freq="W-FRI")
    T = len(dates)
    t5 = lambda: rng.standard_t(5, T) / np.sqrt(5 / 3)          # loi de Student, variance 1
    marche = 0.0025 + 0.022 * t5()
    facteurs = {s: 0.015 * t5() for s in set(secteurs.values())}
    rend = {}
    for t in tickers:
        s = secteurs.get(t, "Marché")
        if t == "SPY":
            r = marche + 0.001 * rng.standard_normal(T)
        elif t == "AGG":
            r = 0.0006 + 0.008 * rng.standard_normal(T) - 0.05 * marche
        elif t.startswith("XL"):
            r = rng.uniform(0.8, 1.2) * marche + facteurs[s] + 0.003 * rng.standard_normal(T)
        else:
            r = (0.0006 * rng.standard_normal() + rng.uniform(0.6, 1.4) * marche
                 + rng.uniform(0.8, 1.5) * facteurs[s] + 0.018 * t5())
        rend[t] = r
    return 100 * (1 + pd.DataFrame(rend, index=dates)).cumprod()


def charger_prix(tickers, secteurs, debut, fin, fichier):
    """Lit le fichier local s'il existe, sinon télécharge sur Yahoo Finance,
    sinon (pas d'accès internet) bascule sur des données simulées."""
    if os.path.exists(fichier):
        prix = pd.read_csv(fichier, index_col=0, parse_dates=True)
        if set(tickers) <= set(prix.columns):
            print(f"Données lues dans {fichier}.")
            return prix[tickers], "fichier local"
    try:
        import yfinance as yf
        prix = yf.download(tickers, start=debut, end=fin, auto_adjust=True, progress=False)["Close"]
        if prix is None or len(prix) == 0 or prix.dropna(how="all").empty:
            raise ValueError("aucune donnée reçue")
        prix = prix[tickers].dropna(how="all")
        manquants = [t for t in tickers if prix[t].isna().mean() > 0.05]
        if prix.empty or manquants:
            raise ValueError(f"séries incomplètes : {manquants}")
        prix.to_csv(fichier)
        print(f"Données téléchargées sur Yahoo Finance et enregistrées dans {fichier}.")
        return prix, "Yahoo Finance"
    except Exception as e:
        print("⚠️  Téléchargement impossible (" + str(e)[:90] + ").")
        print("⚠️  LE NOTEBOOK UTILISE DES DONNÉES SIMULÉES : les chiffres ne décrivent pas les vrais titres.")
        return donnees_simulees(tickers, secteurs, debut, fin), "simulées"


def rendements_hebdo(prix):
    """Rendements hebdomadaires simples, calculés sur les cours du vendredi."""
    return prix.resample("W-FRI").last().pct_change(fill_method=None).dropna(how="any")


def boites(ax, donnees, noms):
    """Boîtes à moustaches, une par colonne de `donnees` (compatible avec toutes les versions de matplotlib)."""
    ax.boxplot(np.asarray(donnees), showfliers=False)
    ax.set_xticks(range(1, len(noms) + 1))
    ax.set_xticklabels(list(noms), rotation=90)


def afficher_matrice(M, titre, vmin=-1, vmax=1, cmap="RdBu_r", fmt="{:.2f}"):
    """Affiche une matrice (DataFrame carré) sous forme de carte de chaleur annotée."""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(M.values, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(range(len(M))); ax.set_xticklabels(M.columns, rotation=90)
    ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
    for i in range(len(M)):
        for j in range(len(M)):
            ax.text(j, i, fmt.format(M.values[i, j]), ha="center", va="center", fontsize=7)
    ax.grid(False); ax.set_title(titre)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.show()

In [ ]:
# Outils d'optimisation (repris du notebook 1)
def tangent(Sigma_, mu_, rf=TAUX_SANS_RISQUE):
    """Portefeuille tangent, vente à découvert permise (formule fermée)."""
    x = np.linalg.solve(Sigma_, mu_ - rf)
    return x / x.sum()

def tangent_positif(Sigma_, mu_, rf=TAUX_SANS_RISQUE):
    """Portefeuille tangent à poids positifs (optimisation numérique)."""
    n = len(mu_)
    res = minimize(lambda w: -(w @ mu_ - rf) / np.sqrt(w @ Sigma_ @ w), np.ones(n) / n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   bounds=[(0, 1)] * n, method="SLSQP", options={"ftol": 1e-10, "maxiter": 500})
    return res.x

def variance_min(Sigma_):
    """Portefeuille de variance minimale, vente à découvert permise."""
    x = np.linalg.solve(Sigma_, np.ones(len(Sigma_)))
    return x / x.sum()

def variance_min_positive(Sigma_):
    """Portefeuille de variance minimale à poids positifs."""
    n = len(Sigma_)
    res = minimize(lambda w: w @ Sigma_ @ w, np.ones(n) / n, jac=lambda w: 2 * Sigma_ @ w,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   bounds=[(0, 1)] * n, method="SLSQP", options={"ftol": 1e-12, "maxiter": 500})
    return res.x

def performance(r, rf=TAUX_SANS_RISQUE):
    """Indicateurs annualisés d'une série de rendements hebdomadaires."""
    valeur = (1 + r).cumprod()
    return pd.Series({"Rendement annuel": r.mean() * SEMAINES,
                      "Volatilité": r.std() * np.sqrt(SEMAINES),
                      "Ratio de Sharpe": (r.mean() * SEMAINES - rf) / (r.std() * np.sqrt(SEMAINES)),
                      "Perte maximale": (valeur / valeur.cummax() - 1).min()})

In [ ]:
prix, SOURCE = charger_prix(TOUS, SECTEURS, DEBUT, FIN, FICHIER)
rendements = rendements_hebdo(prix)
rf = TAUX_SANS_RISQUE / SEMAINES
exces = rendements - rf                    # rendements au-delà du taux sans risque
titres = list(TITRES)
print(f"Source : {SOURCE} — {len(titres)} titres, {len(rendements)} semaines, "
      f"du {rendements.index[0]:%m/%Y} au {rendements.index[-1]:%m/%Y}")

> **Biais du survivant.** Ces 30 titres ont été choisis en 2026 parmi de grandes entreprises toujours cotées.
> Celles qui ont disparu ou décliné entre-temps n'y figurent pas, ce qui embellit les rendements passés.
> Gardez ce biais en tête pour toute la suite.

## 2. Alpha et bêta

Pour chaque titre, on estime sur les dix ans la régression du rendement excédentaire sur celui du marché :

$$R_{i,t} - r_f = \alpha_i + \beta_i\,(R_{M,t} - r_f) + \varepsilon_{i,t}$$

Si le CAPM est vrai, tous les $\alpha_i$ sont nuls.

### Exercice 2.1
Écrivez une fonction `regression(y, x)` qui renvoie $\alpha$, $\beta$ et leurs statistiques $t$.
Appliquez-la aux 30 titres, et comptez les alphas significatifs au seuil de 5 % ($|t| > 1{,}96$).

In [ ]:
def regression(y, x):
    """Moindres carrés de y sur une constante et x. Renvoie alpha, bêta, t(alpha), t(bêta), R²."""
    # À COMPLÉTER
    ...

def table_capm(ex, ex_marche):
    lignes = {}
    for t in titres:
        a, b, ta, tb, r2 = regression(ex[t].values, ex_marche.values)
        lignes[t] = {"Secteur": TITRES[t], "Alpha annuel": a * SEMAINES, "t (alpha)": ta, "Bêta": b, "R²": r2}
    return pd.DataFrame(lignes).T.astype({"Alpha annuel": float, "t (alpha)": float, "Bêta": float, "R²": float})

capm = table_capm(exces, exces[MARCHE])
# À COMPLÉTER : nombre d'alphas significatifs
n_sig = ...

In [ ]:
plt.hist(capm["t (alpha)"], bins=15, edgecolor="white")
for s in (-1.96, 1.96):
    plt.axvline(s, color="C3", ls="--")
plt.title("Statistiques t des alphas : au-delà des lignes rouges, alpha « significatif » à 5 %")
plt.xlabel("t (alpha)"); plt.show()

## 3. Tester la droite de marché des titres

Le test classique se fait en deux temps, pour éviter d'utiliser les mêmes données deux fois :

1. On estime les bêtas sur la **première moitié** (2015-2019).
2. On mesure les rendements excédentaires moyens sur la **seconde moitié** (2020-2024), et on régresse ces moyennes
   sur les bêtas : $\bar R_i - r_f = \gamma_0 + \gamma_1\, \beta_i$.

Le CAPM prédit $\gamma_0 = 0$ et $\gamma_1$ égal à la prime de marché réalisée.

### Exercice 3.1
Réalisez ce test et tracez le nuage de points avec la droite théorique et la droite estimée.

In [ ]:
p1, p2 = exces.loc["2015":"2019"], exces.loc["2020":"2024"]
# À COMPLÉTER
betas_1 = ...      # bêtas estimés sur 2015-2019
moyennes_2 = ...   # rendements excédentaires moyens annualisés sur 2020-2024
prime_2 = ...      # prime de marché réalisée sur 2020-2024
gamma_1, gamma_0 = ...

### 3.2 Des portefeuilles triés par bêta

Pour réduire le bruit, on regroupe les titres en trois portefeuilles équipondérés selon leur bêta de 2015-2019,
puis on mesure leurs performances sur 2020-2024.

In [ ]:
groupes = pd.qcut(betas_1, 3, labels=["Bêta faible", "Bêta moyen", "Bêta élevé"])
lignes = {}
for g in groupes.cat.categories:
    membres = groupes[groupes == g].index
    r = rendements.loc["2020":"2024", membres].mean(axis=1)
    a, b, *_ = regression(r.values - rf, p2[MARCHE].values)
    lignes[g] = {"Bêta 2015-2019": betas_1[membres].mean(), "Bêta réalisé 2020-2024": b, **performance(r).to_dict()}
tri_beta = pd.DataFrame(lignes).T
tri_beta.style.format({"Bêta 2015-2019": "{:.2f}", "Bêta réalisé 2020-2024": "{:.2f}", "Rendement annuel": "{:.1%}",
                       "Volatilité": "{:.1%}", "Ratio de Sharpe": "{:.2f}", "Perte maximale": "{:.1%}"})

## 4. Les hypothèses à l'épreuve

### 4.1 Des rendements gaussiens ?

Le CAPM raisonne en moyenne et variance, ce qui est pleinement justifié si les rendements suivent une loi normale.
On mesure l'asymétrie (*skewness*, 0 pour une loi normale) et l'aplatissement en excès (*kurtosis*, 0 pour une loi normale),
et on applique le test de Jarque-Bera.

### Exercice 4.1
Calculez ces statistiques pour le marché et les 30 titres. Quelle est la pire semaine du S&P 500,
et à quelle fréquence une loi normale prévoirait-elle une telle chute ?

In [ ]:
r = rendements[titres + [MARCHE]]
# À COMPLÉTER : asymétrie, aplatissement, p-valeur de stats.jarque_bera pour chaque série
normalite = ...

m = rendements[MARCHE]
# À COMPLÉTER : pire semaine, nombre d'écarts-types, probabilité selon la loi normale (stats.norm.cdf)
...

### 4.2 Des bêtas stables ?

Le CAPM traite le bêta comme une constante. On l'estime sur une fenêtre glissante de deux ans.

In [ ]:
fenetre = 104
exemples = ["AAPL", "JPM", "XOM", "KO", "NEE"]
betas_glissants = pd.DataFrame({t: rendements[t].rolling(fenetre).cov(rendements[MARCHE])
                                   / rendements[MARCHE].rolling(fenetre).var() for t in exemples}).dropna()
betas_glissants.plot(title="Bêtas estimés sur deux ans glissants")
plt.axhline(1, color="black", lw=0.8); plt.ylabel("Bêta"); plt.show()

### 4.3 Quel portefeuille de marché ? La critique de Roll (1977)

Le vrai portefeuille de marché contient tous les actifs : actions du monde entier, obligations, immobilier,
et même le capital humain. Il n'est pas observable. Le S&P 500 n'en est qu'un substitut imparfait.

### Exercice 4.3
Recalculez les bêtas et les alphas avec un autre substitut du marché, 60 % d'actions (`SPY`) et 40 % d'obligations (`AGG`).
Le classement des titres par bêta change-t-il ? Et le nombre d'alphas significatifs ?

In [ ]:
marche_6040 = 0.6 * rendements[MARCHE] + 0.4 * rendements[OBLIGATIONS] - rf
# À COMPLÉTER : table_capm avec ce nouveau marché, corrélation des classements (stats.spearmanr)
...

### 4.4 Emprunter sans limite au taux sans risque ?

C'est l'hypothèse la plus fragile. Un particulier emprunte difficilement pour investir, et à un taux supérieur
au taux sans risque. Fischer Black (1972) a montré que, si l'emprunt est contraint, la droite de marché
devient **plus plate** : les investisseurs qui veulent plus de rendement sans pouvoir s'endetter achètent
des titres à fort bêta, ce qui fait monter leur prix et baisser leur rendement futur.

Celui qui peut emprunter dispose alors d'une stratégie : acheter les titres à faible bêta et les mettre
sous levier pour retrouver un bêta de 1. C'est la logique de la parité de risque.

### Exercice 4.4
Sur 2020-2024, comparez le portefeuille à bêta élevé et le portefeuille à bêta faible mis sous levier
(emprunt au taux sans risque) pour atteindre un bêta de 1.

In [ ]:
faible = rendements.loc["2020":"2024", groupes[groupes == "Bêta faible"].index].mean(axis=1)
eleve = rendements.loc["2020":"2024", groupes[groupes == "Bêta élevé"].index].mean(axis=1)
beta_faible = regression(faible.values - rf, p2[MARCHE].values)[1]
beta_eleve = regression(eleve.values - rf, p2[MARCHE].values)[1]
levier = 1 / beta_faible
faible_levier = rf + levier * (faible - rf)
eleve_ajuste = rf + (eleve - rf) / beta_eleve
print(f"Levier appliqué au portefeuille à bêta faible : {levier:.2f}")
pd.DataFrame({"Bêta faible, levier → bêta 1": performance(faible_levier),
              "Bêta élevé, ramené à bêta 1": performance(eleve_ajuste),
              "Marché": performance(rendements.loc["2020":"2024", MARCHE])}).T.style.format(
    {"Rendement annuel": "{:.1%}", "Volatilité": "{:.1%}", "Ratio de Sharpe": "{:.2f}", "Perte maximale": "{:.1%}"})

### 4.5 Les autres hypothèses

- **Un horizon unique et des préférences moyenne-variance.** Les investisseurs réels ont des horizons différents
  et craignent davantage certaines pertes que d'autres : perdre au moment où l'on perd aussi son emploi, par exemple.
  Le CAPM intertemporel de Merton (1973) et les modèles fondés sur la consommation en tiennent compte.
- **Des anticipations identiques.** Si les investisseurs ne sont pas d'accord, chacun détient un portefeuille
  différent, et le marché n'est plus que leur moyenne pondérée.
- **Le capital humain.** David Mayers (1972) intègre au CAPM les revenus du travail, non négociables.
  Chacun devrait alors ajuster le portefeuille de marché pour compenser son propre capital humain :
  plus d'actions pour un fonctionnaire au revenu stable, plus d'obligations pour un salarié dont les revenus
  dépendent des marchés.
- **Pas de frais ni d'impôts.** Les coûts de transaction et la fiscalité rendent la rotation coûteuse
  et favorisent les stratégies passives.

## 5. L'arithmétique de la gestion active (Sharpe, 1991)

En 1991, Sharpe énonce un argument qui ne repose sur **aucune** des hypothèses du CAPM, mais sur une identité comptable.
Les gérants passifs détiennent le marché. Les gérants actifs, pris ensemble, détiennent donc aussi le marché.
Avant frais, le gérant actif moyen (pondéré par les capitaux) fait donc exactement le rendement du marché ;
après frais, il fait moins bien.

### Exercice 5.1
Simulez 200 gérants actifs de même taille qui s'écartent au hasard du marché (ici, le portefeuille équipondéré de nos
30 titres), de sorte que leur position agrégée soit exactement le marché. Comparez leurs performances avant
et après des frais de 1 % par an, à celle d'un fonds passif facturant 0,05 %.

In [ ]:
rng = np.random.default_rng(11)
R30 = rendements[titres]
n = len(titres)
w_marche = np.ones(n) / n
# À COMPLÉTER : 100 paires de gérants aux écarts opposés, rendements avant et après frais, comparaison au passif
...

## 6. Ce qu'il en reste

| Ce que dit le CAPM | Ce que montrent les données |
|---|---|
| Seul le bêta est rémunéré | La droite de marché est trop plate ; d'autres facteurs comptent (taille, valeur, rentabilité) |
| Les alphas sont nuls | Quelques alphas significatifs, autant qu'en attendrait le hasard sur un petit échantillon |
| Rendements décrits par moyenne et variance | Queues épaisses, krachs bien plus fréquents qu'une loi normale |
| Bêtas constants | Bêtas instables dans le temps |
| Le portefeuille de marché est efficient | Inobservable : tout test dépend du substitut choisi (Roll) |

Le CAPM est faux au sens strict, et Sharpe le sait mieux que personne. Mais dans *Investors and Markets* (2007),
il défend une version qui ne dépend plus de ces hypothèses : le portefeuille de marché est la position de
**l'investisseur moyen**, la seule que tout le monde peut détenir en même temps. S'en écarter est légitime,
mais suppose de savoir en quoi l'on diffère de cet investisseur moyen : son horizon, son capital humain, ses contraintes
d'emprunt, ou une information que les autres n'ont pas.

Ce qui survit à toutes les critiques tient en quelques principes simples : des frais bas, une diversification
large entre classes d'actifs et à l'intérieur de chacune, un niveau de risque total adapté à sa situation,
et la discipline de s'y tenir.

**Pour aller plus loin**

1. Refaites le test de la section 3 en inversant les périodes : bêtas sur 2020-2024, rendements sur 2015-2019.
2. Ajoutez les facteurs de Fama et French à la régression de la section 2. Les alphas diminuent-ils ?
3. Lisez Sharpe, « The Arithmetic of Active Management », *Financial Analysts Journal*, 1991 : trois pages.